# Limpieza de datos

En el notebook anterior (`01_exploracion_calidad_ml.ipynb`) analizamos la calidad de los datos y definimos los criterios de limpieza en la tabla 3.6. Acá aplicamos esas decisiones a las tablas de usuarios, características de audio y escuchas.

Guardamos las tablas limpias en formato Parquet para usarlas en los siguientes notebooks sin repetir la limpieza.

Los resultados y las observaciones con números corresponden a la ejecución original. Si volvemos a ejecutar las consultas, tenemos que comprobar que coincidan.

## 1. Preparación del entorno

Usamos DuckDB porque nos permite consultar los archivos Parquet con SQL sin cargar todas las escuchas en un DataFrame. Configuramos un límite de memoria de 4 GB para DuckDB y una carpeta para los archivos temporales. Esto no implica que todo el proceso use como máximo 4 GB de RAM.

Creamos una vista para cada archivo original. Las vistas no copian los datos: nos permiten consultar los archivos por su nombre. La función `q()` ejecuta una consulta y devuelve el resultado como un DataFrame, que usamos para los controles.

Las rutas son relativas a la carpeta de trabajo. Hay que ejecutar el notebook desde la misma carpeta que los anteriores, con `data/` un nivel por encima.

In [ ]:
import os
import duckdb

con = duckdb.connect()
con.execute("SET memory_limit = '4GB'")
con.execute("SET TimeZone = 'UTC'")
con.execute("SET temp_directory = '../.tmp'")

# ../ apunta a la carpeta anterior a la de trabajo
os.makedirs("../data/clean", exist_ok=True)

ruta = "../data/parquet/"
con.execute(f"CREATE OR REPLACE VIEW events AS SELECT * FROM '{ruta}events.parquet'")
con.execute(f"CREATE OR REPLACE VIEW users AS SELECT * FROM '{ruta}users.parquet'")
con.execute(f"CREATE OR REPLACE VIEW acoustic_features AS SELECT * FROM '{ruta}acoustic_features.parquet'")

def q(consulta):
    return con.sql(consulta).df()

q("SELECT 'todo listo' AS estado")

## 2. Limpieza de las tablas

Aplicamos las decisiones de la tabla 3.6 del notebook 01. No eliminamos usuarios ni canciones por los valores fuera de rango: los reemplazamos por nulos. En la tabla de escuchas sacamos la de 1970 y las de los días en que un usuario superó las 720 escuchas, según el criterio que definimos en el análisis anterior.

Usamos `COPY ... TO` para guardar directamente el resultado de cada consulta en un archivo Parquet, sin pasarlo primero a Pandas.

### 2.1 Usuarios

Consideramos válidas las edades entre 10 y 90 años. Las que quedan afuera de ese rango, incluido el -1, las pasamos a nulo. En género conservamos `m` y `f`, y el resto queda como "no informado". También unificamos UK y GB usando GB para el Reino Unido.

Para hacer estos cambios usamos `CASE WHEN`, que funciona de manera parecida a un `if`: si se cumple la condición, devuelve el valor indicado después de `THEN`; si no, devuelve el de `ELSE`.

In [2]:
con.execute("""
    COPY (
        SELECT user_id,
               CASE WHEN country = 'UK' THEN 'GB' ELSE country END        AS country,
               CASE WHEN age BETWEEN 10 AND 90 THEN age ELSE NULL END       AS age,
               CASE WHEN gender IN ('m', 'f') THEN gender
                    ELSE 'no informado' END                                AS gender,
               playcount,
               registered_unixtime
        FROM users
    ) TO '../data/clean/users_clean.parquet' (FORMAT parquet)
""")

# Revisamos cómo quedaron las columnas que modificamos
q("""
    SELECT COUNT(*)                                    AS usuarios,
           COUNT(*) FILTER (WHERE age IS NULL)          AS edad_nula,
           COUNT(*) FILTER (WHERE gender = 'no informado') AS genero_no_informado,
           COUNT(*) FILTER (WHERE country = 'UK')       AS quedan_uk
    FROM '../data/clean/users_clean.parquet'
""")

,usuarios,edad_nula,genero_no_informado,quedan_uk
0,55190,12481,4747,0


Quedaron los 55.190 usuarios. De ellos, 12.481 tienen la edad en nulo y 4.747 tienen el género como "no informado". Ya no quedan casos con el código UK.

### 2.2 Características de audio

Pasamos a nulo los valores de `tempo` iguales a 0 y los de `loudness` iguales a -60 dB, como decidimos en el notebook anterior. Los tratamos como valores que no queremos utilizar para estos análisis, sin borrar las canciones.

Dejamos afuera `key` (nota musical) y `mode` (modo mayor o menor) porque no las vamos a utilizar en los modelos. Son variables categóricas, aunque estén guardadas como números.

Con `NULLIF` podemos reemplazar un valor puntual por un nulo. Por ejemplo, `NULLIF(tempo, 0)` devuelve nulo si el tempo es 0 y, si no, conserva el valor original.

In [3]:
con.execute("""
    COPY (
        SELECT track_id,
               danceability,
               energy,
               NULLIF(loudness, -60) AS loudness,
               speechiness,
               acousticness,
               instrumentalness,
               liveness,
               valence,
               NULLIF(tempo, 0)      AS tempo
        FROM acoustic_features
    ) TO '../data/clean/acoustic_clean.parquet' (FORMAT parquet)
""")

# COUNT(columna) no cuenta los nulos
q("""
    SELECT COUNT(*)                  AS canciones,
           COUNT(*) - COUNT(tempo)    AS tempo_nulo,
           COUNT(*) - COUNT(loudness) AS loudness_nulo
    FROM '../data/clean/acoustic_clean.parquet'
""")

,canciones,tempo_nulo,loudness_nulo
0,3471884,2651,209


Quedaron 2.651 registros con tempo nulo y 209 con loudness nulo. Coincide con lo observado en el notebook 01.

### 2.3 Escuchas

Eliminamos las escuchas anteriores al 1/1/2005 (en nuestros datos, la de 1970) y todas las escuchas de los días en que un usuario superó las 720. Este último es el umbral que definimos en el análisis de calidad para detectar actividad anómala.

Para agrupar las escuchas por día hacemos una división entera del timestamp por 86.400, que es la cantidad de segundos de un día. Así obtenemos el número de día en UTC sin tener que convertir cada timestamp a una fecha. El número 1104537600 corresponde al 1/1/2005 a las 00:00 UTC.

Dividimos la consulta en partes con `WITH`:

- `dias`: cuenta las escuchas de cada usuario por día.
- `dias_anomalos`: selecciona los días con más de 720 escuchas.

Después hacemos un `LEFT JOIN` entre las escuchas originales y los días anómalos usando el usuario y el día. Si una escucha no pertenece a un día anómalo, `a.user_id` queda nulo. Con `WHERE a.user_id IS NULL` nos quedamos con esas escuchas y, además, filtramos las fechas anteriores a 2005.

Por ejemplo, si un usuario tuvo 800 escuchas el lunes y 30 el martes, eliminamos las del lunes pero conservamos las del martes.

In [4]:
con.execute("""
    COPY (
        -- Escuchas por usuario y día
        WITH dias AS (
            SELECT user_id, "timestamp" // 86400 AS dia, COUNT(*) AS escuchas_dia
            FROM events
            GROUP BY user_id, dia
        ),
        -- Días que superan el límite
        dias_anomalos AS (
            SELECT user_id, dia
            FROM dias
            WHERE escuchas_dia > 720
        )
        -- Conservamos las escuchas que pasan los filtros
        SELECT e.user_id, e.artist_id, e.album_id, e.track_id, e."timestamp"
        FROM events e
        LEFT JOIN dias_anomalos a
               ON e.user_id = a.user_id
              AND e."timestamp" // 86400 = a.dia
        WHERE a.user_id IS NULL
          AND e."timestamp" >= 1104537600
    ) TO '../data/clean/events_clean.parquet' (FORMAT parquet, COMPRESSION zstd)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### 2.4 Control de la limpieza

Comparamos la cantidad de escuchas antes y después de la limpieza para saber qué porcentaje de los datos eliminamos. `fetchone()[0]` obtiene el primer valor que devuelve la consulta (en este caso, el total de filas).

In [5]:
antes   = con.execute("SELECT COUNT(*) FROM events").fetchone()[0]
despues = con.execute("SELECT COUNT(*) FROM '../data/clean/events_clean.parquet'").fetchone()[0]

print(f"Escuchas originales: {antes:>15,}")
print(f"Escuchas limpias:    {despues:>15,}")
print(f"Eliminadas:          {antes - despues:>15,}  ({(antes - despues) / antes * 100:.2f}%)")

Escuchas originales:     351,469,333
Escuchas limpias:        350,741,953
Eliminadas:                  727,380  (0.21%)


Se eliminaron 727.380 escuchas, que representan el 0,21% del total. Según lo calculado en el notebook anterior, 727.379 corresponden a días anómalos y una a la fecha de 1970.



Los archivos que quedan guardados en `data/clean/` son:\
 `users_clean.parquet`\
 `acoustic_clean.parquet`\
 `events_clean.parquet` 